In [1]:
# Track 1 reliability eval: Model A (ResNet-50) + Model B (ViT-B/16) on OASIS test scans
import os, glob, subprocess, sys, json
import pandas as pd
os.chdir('/kaggle/working')
if not os.path.isdir('Neuroshield'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/pankurajasaivardhan/Neuroshield.git'], check=True)
ROOT = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/splits/train.csv', recursive=True)[0]))
print('dataset ROOT:', ROOT)
OUT_DATA = '/kaggle/working/oasis'
os.makedirs(f'{OUT_DATA}/splits', exist_ok=True)
for s in ('train', 'val', 'test'):
    df = pd.read_csv(f'{ROOT}/splits/{s}.csv')
    df['scan_path'] = df['scan_path'].apply(lambda x: os.path.join(ROOT, 'data', str(x).split('../../data/oasis/')[-1]))
    missing = [p for p in df['scan_path'] if not os.path.exists(p)]
    assert not missing, f'{s}: {len(missing)} missing scans, e.g. {missing[:2]}'
    df.to_csv(f'{OUT_DATA}/splits/{s}.csv', index=False)
    print(s, len(df), 'scans ok')
print('A:', glob.glob('/kaggle/input/**/model_a_resnet50_seed42_best.pt', recursive=True))
print('B:', glob.glob('/kaggle/input/**/model_b_vit_seed42_best.pt', recursive=True))


Cloning into 'Neuroshield'...


dataset ROOT: /kaggle/input/datasets/mohanroyal273/neuroshield-oasis-model-b
train 164 scans ok
val 35 scans ok
test 36 scans ok
A: ['/kaggle/input/notebooks/mohanroyal273/track1-model-a-resnet50-baseline/outputs/track1/model_a/model_a_resnet50_seed42_best.pt']
B: ['/kaggle/input/notebooks/mohanroyal273/track1-model-a-resnet50-baseline/neuroshield_repo/outputs/alzheimers/model_b/checkpoints/model_b_vit_seed42_best.pt', '/kaggle/input/notebooks/mohanroyal273/03-train-model-b-final/model_b_vit_seed42_best.pt']


In [2]:
import subprocess, os, glob, json, sys
os.chdir('/kaggle/working/Neuroshield')
r = subprocess.run([sys.executable, '-m', 'track1_alzheimers.src.eval_reliability',
                    '--data-root', '/kaggle/working/oasis',
                    '--model-a', glob.glob('/kaggle/input/**/model_a_resnet50_seed42_best.pt', recursive=True)[0],
                    '--model-b', [p for p in glob.glob('/kaggle/input/**/model_b_vit_seed42_best.pt', recursive=True) if '03-train-model-b-final' in p][0],
                    '--out', '/kaggle/working/outputs'])
print('exit', r.returncode)
assert r.returncode == 0, 'reliability eval failed - see log'
res = json.load(open('/kaggle/working/outputs/reliability_results.json'))
for name, splits in res['models'].items():
    print(name, json.dumps(splits['test'], indent=1))
print('gate:', json.dumps(res['gate']['thresholds'], indent=1))


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 253MB/s]


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 256MB/s]


[model_a] {
  "train": {
    "ece_raw": 0.20073233951882613,
    "ece_calibrated": 0.20073233951882613,
    "brier": 0.16623216722143971,
    "accuracy": 0.948780487804878,
    "mean_ood": 816.9204181733235
  },
  "val": {
    "ece_raw": 0.0691389046396528,
    "ece_calibrated": 0.07167525904519219,
    "brier": 0.4284745658821948,
    "accuracy": 0.6742857142857143,
    "mean_ood": 9036971.736767936
  },
  "test": {
    "ece_raw": 0.09147700170675914,
    "ece_calibrated": 0.08290421134895747,
    "brier": 0.4357642622066646,
    "accuracy": 0.6111111111111112,
    "mean_ood": 8492485.186425362
  }
}
[model_b] {
  "train": {
    "ece_raw": 0.07834170402550117,
    "ece_calibrated": 0.07834170402550117,
    "brier": 0.4850732624078604,
    "accuracy": 0.6012195121951219,
    "mean_ood": 736.8796755223635
  },
  "val": {
    "ece_raw": 0.11639367478234428,
    "ece_calibrated": 0.07653906958443775,
    "brier": 0.4720262973493821,
    "accuracy": 0.5771428571428572,
    "mean_ood": 1031